Gerekli importlar

In [2]:
import requests
import re
from youtube_comment_downloader import *
import pandas as pd
import itertools
import time

In [3]:
video_linkleri = [
    'https://www.youtube.com/watch?v=0vf80JFe4tU', # Tuğba Yılmaz - Ev Pizzası Tarifi
    'https://www.youtube.com/watch?v=VAO5geOPUwI', # Arapaşı - atakancemile
    'https://www.youtube.com/watch?v=2Ne1D3dIJ5M', # Enis Arıkan - garaj 
    'https://www.youtube.com/watch?v=ew39U719B2c', # Enis Kirazoğlu - Yemekfinal
    'https://www.youtube.com/watch?v=q9NbnZa0FJw', # cem yılmaz standup 
    'https://www.youtube.com/watch?v=57gh5GlkzTg' # orkun ışıtmak - EN DÜŞÜK PUANLI YURT DIŞI TURU
]

In [4]:
downloader = YoutubeCommentDownloader()
tum_yorum_verileri = []

In [5]:
print(f"{len(video_linkleri)} adet video taranıyor...\n")

6 adet video taranıyor...



In [6]:
# --- VİDEO BAŞLIĞINI ÇEKEN YARDIMCI FONKSİYON ---
def video_basligini_al(url):
    try:
        # Videonun web sayfasına ufak bir istek atıyoruz
        response = requests.get(url)
        # HTML kodları içindeki <title> etiketini buluyoruz
        baslik = re.search(r'<title>(.*?)</title>', response.text).group(1)
        # Başlığın sonundaki " - YouTube" yazısını temizliyoruz
        return baslik.replace(" - YouTube", "").strip()
    except Exception:
        return "Bilinmeyen Video"

In [7]:
# 2. Her bir link için döngü başlatıyoruz
for index, url in enumerate(video_linkleri, 1):
    try:
        print(f"[{index}/{len(video_linkleri)}] Veri çekiliyor: {url}")
        
        # ÖNCE VİDEO BAŞLIĞINI ÇEKİYORUZ
        video_adi = video_basligini_al(url)
        print(f"[{index}/{len(video_linkleri)}] {video_adi}")
        print(f"Yorumlar çekiliyor, lütfen bekleyin...")
        
        # İŞTE EKSİK OLAN VE SENİN SİLDİĞİN SATIR BURASI:
        # Videodan yorumları çek (Popülerliğe göre)
        comments = downloader.get_comments_from_url(url, sort_by=SORT_BY_POPULAR)
        
        # Hafıza değişkenlerimiz (Ana yorumları akılda tutmak için)
        ana_yorum_id = None
        ana_yorum_yazari = None
        
        video_sayaci = 0
        
        # Her videodan kaç yorum alınsın? (Şu an 2000)
        for comment in itertools.islice(comments, 2000):
            is_reply = comment.get('reply', False)
            yorum_id = comment.get('cid', 'Bilinmiyor')
            yazar = comment.get('author', 'Bilinmiyor')
            
            # YANIT KONTROLÜ VE HAFIZA İŞLEMİ
            if not is_reply:
                # Bu bir "Ana Yorum". Bilgilerini hafızaya alıyoruz.
                ana_yorum_id = yorum_id
                ana_yorum_yazari = yazar
                yanit_edilen_kisi = None
                ust_yorum_id = None
            else:
                # Bu bir "Yanıt". Hafızadaki son ana yoruma aittir.
                yanit_edilen_kisi = ana_yorum_yazari
                ust_yorum_id = ana_yorum_id
            
            # Verileri listeye ekle
            tum_yorum_verileri.append({
                'Video Başlığı': video_adi,
                'Video Kaynağı': url,
                'Yorum ID': yorum_id,
                'Üst Yorum ID': ust_yorum_id,
                'Yorum Yapan Kişi': yazar,
                'Yanıtlanan Kişi': yanit_edilen_kisi,
                'Yorum Metni': comment.get('text', ''),
                'Beğeni Sayısı': comment.get('votes', 0),
                'Tarih': comment.get('time', 'Bilinmiyor'),
                'Yanıt Mı?': 'Evet' if is_reply else 'Hayır'
            })
            video_sayaci += 1
        
        print(f"Başarılı! Bu videodan {video_sayaci} yorum alındı.\n")
        
        # YouTube bot korumasına takılmamak için kısa bir bekleme
        time.sleep(1) 
        
    except Exception as e:
        print(f"Hata oluştu ({url}): {e}\n")

[1/6] Veri çekiliyor: https://www.youtube.com/watch?v=0vf80JFe4tU
[1/6] AÇIK MUTFAK - EV PİZZASI @Tugbayiilmaaz
Yorumlar çekiliyor, lütfen bekleyin...
Başarılı! Bu videodan 1200 yorum alındı.

[2/6] Veri çekiliyor: https://www.youtube.com/watch?v=VAO5geOPUwI
[2/6] AÇIK MUTFAK - ARABAŞI ÇORBASI @AtakanCemile
Yorumlar çekiliyor, lütfen bekleyin...
Başarılı! Bu videodan 1032 yorum alındı.

[3/6] Veri çekiliyor: https://www.youtube.com/watch?v=2Ne1D3dIJ5M
[3/6] Enis Arıkan | Melis İşiten ile Zaten Şov - 138. Bölüm
Yorumlar çekiliyor, lütfen bekleyin...
Başarılı! Bu videodan 1083 yorum alındı.

[4/6] Veri çekiliyor: https://www.youtube.com/watch?v=ew39U719B2c
[4/6] YEMEK İÇİN GELDİK, EVE ALMADILAR! ELRAEN, ERAY, LIMONIFY, KONSOL OYUN, RRAENEE
Yorumlar çekiliyor, lütfen bekleyin...
Başarılı! Bu videodan 1882 yorum alındı.

[5/6] Veri çekiliyor: https://www.youtube.com/watch?v=q9NbnZa0FJw
[5/6] Cem Yılmaz | Melis İşiten ile Zaten Şov - 123. Bölüm
Yorumlar çekiliyor, lütfen bekleyin...
Başarıl

In [8]:
# 3. Tüm verileri Pandas ile tabloya çevirip kaydediyoruz
df = pd.DataFrame(tum_yorum_verileri)

In [9]:
# Türkçe karakter desteğiyle kaydet (Excel'de bozulmaması için utf-8-sig)
df.to_csv('toplu_yorum_verileri.csv', index=False, encoding='utf-8-sig')

In [10]:
print("="*40)
print(f"İşlem Tamamlandı!")
print(f"Toplam çekilen yorum sayısı: {len(df)}")
print(f"Dosya adı: toplu_yorum_verileri.csv")
print("="*40)

İşlem Tamamlandı!
Toplam çekilen yorum sayısı: 7934
Dosya adı: toplu_yorum_verileri.csv


In [11]:
df

,Video Başlığı,Video Kaynağı,Yorum ID,Üst Yorum ID,Yorum Yapan Kişi,Yanıtlanan Kişi,Yorum Metni,Beğeni Sayısı,Tarih,Yanıt Mı?
0,AÇIK MUTFAK - EV PİZZASI @Tugbayiilmaaz,https://www.youtube.com/watch?v=0vf80JFe4tU,UgyJ0kV7TGCEACL_PDF4AaABAg,None,@Tugbayiilmaaz,None,AÇIK MUTFAK: ÇOCUKLUK TRAVMALARIM..,6 B,1 yıl önce,Hayır
1,AÇIK MUTFAK - EV PİZZASI @Tugbayiilmaaz,https://www.youtube.com/watch?v=0vf80JFe4tU,UgxvqvoUsfM4vj9LFeZ4AaABAg,None,@satbaskn2122,None,Melisten sonra tuğbayı izlemeye geldim,1 B,2 ay önce,Hayır
2,AÇIK MUTFAK - EV PİZZASI @Tugbayiilmaaz,https://www.youtube.com/watch?v=0vf80JFe4tU,UgzlvgeWqQFDB8YUokF4AaABAg,None,@winwin-222,None,Psikolog : 4000 tl ❌\nAçık mutfak : ücretsiz ✅,"1,6 B",1 yıl önce,Hayır
3,AÇIK MUTFAK - EV PİZZASI @Tugbayiilmaaz,https://www.youtube.com/watch?v=0vf80JFe4tU,UgzXIfEfHrhuEOIphG14AaABAg,None,@elifs6656,None,Tuğba'nın gülüşünü gormeye geldim kdndkdndkdn,637,2 ay önce,Hayır
4,AÇIK MUTFAK - EV PİZZASI @Tugbayiilmaaz,https://www.youtube.com/watch?v=0vf80JFe4tU,UgzzjXXFmU5Lb8bYywF4AaABAg,None,@elif.elif153,None,Bidaha bidaha tugbayla cekin,358,6 ay önce,Hayır
...,...,...,...,...,...,...,...,...,...,...
7929,EN DÜŞÜK PUANLI YURT DIŞI TURU!,https://www.youtube.com/watch?v=57gh5GlkzTg,UgwluReN0KKaYRQmG7N4AaABAg.AW6ICGZrWftAW6IN_N4un2,UgyVhZxcjNjZgnIT2uZ4AaABAg,@Selamm45m,@muratozturk7678,6 yaşında okula başladıysa sınıfındakıler 3 ya...,0,3 hafta önce,Evet
7930,EN DÜŞÜK PUANLI YURT DIŞI TURU!,https://www.youtube.com/watch?v=57gh5GlkzTg,UgwluReN0KKaYRQmG7N4AaABAg.AW6ICGZrWftAW6MErRJiHm,UgyVhZxcjNjZgnIT2uZ4AaABAg,@GuideBalthazar,@muratozturk7678,"Merhaba, ben turun rehberi Ahmet Sait BULUT\n\...",0,3 hafta önce,Evet
7931,EN DÜŞÜK PUANLI YURT DIŞI TURU!,https://www.youtube.com/watch?v=57gh5GlkzTg,UgymRJOYPcWRPHEmo4F4AaABAg.AW33DOwioDeAW3RR0gtd3F,UgyVhZxcjNjZgnIT2uZ4AaABAg,@dennisadyaman963,@muratozturk7678,Adam hakli,0,3 hafta önce,Evet
7932,EN DÜŞÜK PUANLI YURT DIŞI TURU!,https://www.youtube.com/watch?v=57gh5GlkzTg,UgyVhZxcjNjZgnIT2uZ4AaABAg.AW1cmoLBh2KAWx3LRM7nTc,UgyVhZxcjNjZgnIT2uZ4AaABAg,@nfurkann,@muratozturk7678,Sorunlusun,2,4 gün önce,Evet


In [12]:
# İlk satırdaki (0. index) 'Yorum Metni' sütununun tamamını yazdırır
print(df['Yorum Metni'].iloc[0])

AÇIK MUTFAK: ÇOCUKLUK TRAVMALARIM..
